# 💠 ClearSight V6: Enterprise Retroactive Re-Identification Engine

**Designed for High-Stakes CCTV Analysis**

This architecture abandons frame-by-frame analysis in favor of **Tracklet-Based Deep ReID** combined with **Statistical Anomaly Auto-Thresholding**. 
By tracking a person's entire lifetime in the video (a "Tracklet") and finding the single best frame of their face, we can completely eliminate false positives. Once verified, we mathematically trace their path backwards to the very second they entered the frame.

### Phase 1: Initialize SOTA Neural Networks
We use **InsightFace (ArcFace ResNet)** for identity verification, which is the current industry standard in high-end surveillance.

In [1]:
import cv2
import torch
import numpy as np
from ultralytics import YOLO
import insightface
from insightface.app import FaceAnalysis
import matplotlib.pyplot as plt

print("Initializing V6 Engine...")

# 1. YOLOv8 for Body Tracking
yolo_model = YOLO('yolov8n.pt')

# 2. InsightFace ArcFace for SOTA Identity Verification
# Note: 'buffalo_l' is the state-of-the-art model pack containing RetinaFace and ArcFace ResNet50
face_app = FaceAnalysis(name='buffalo_l', providers=['CUDAExecutionProvider', 'CPUExecutionProvider'])
face_app.prepare(ctx_id=0, det_size=(640, 640))
print("Neural Networks Loaded Perfectly.")

Initializing V6 Engine...


c:\Users\rajti\raj123\envs\clearsight\lib\site-packages\onnxruntime\capi\onnxruntime_inference_collection.py:69: UserWarning: Specified provider 'CUDAExecutionProvider' is not in available provider names.Available providers: 'AzureExecutionProvider, CPUExecutionProvider'
  warnings.warn(


Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: C:\Users\rajti/.insightface\models\buffalo_l\1k3d68.onnx landmark_3d_68 ['None', 3, 192, 192] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: C:\Users\rajti/.insightface\models\buffalo_l\2d106det.onnx landmark_2d_106 ['None', 3, 192, 192] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: C:\Users\rajti/.insightface\models\buffalo_l\det_10g.onnx detection [1, 3, '?', '?'] 127.5 128.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: C:\Users\rajti/.insightface\models\buffalo_l\genderage.onnx genderage ['None', 3, 96, 96] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: C:\Users\rajti/.insightface\models\buffalo_l\w600k_r50.onnx recognition ['None', 3, 112, 112] 127.

### Phase 2: Generate the Master Identity Vector
We extract the deep embeddings from the police reference photos.

In [2]:
target_image_paths = [
    'C:/Users/rajti/Downloads/srk1.webp',
    'C:/Users/rajti/Downloads/srk2.jpg',
    'C:/Users/rajti/Downloads/srk3.jpg'
]

embeddings = []
for path in target_image_paths:
    img = cv2.imread(path)
    faces = face_app.get(img)
    if len(faces) > 0:
        # Get the largest face in the photo
        biggest_face = max(faces, key=lambda f: (f.bbox[2]-f.bbox[0])*(f.bbox[3]-f.bbox[1]))
        embeddings.append(biggest_face.embedding)
        print(f"Extracted vector from {path}")

# Calculate the Master Vector by averaging all provided photos
master_vector = np.mean(embeddings, axis=0)
master_vector = master_vector / np.linalg.norm(master_vector) # Normalize
print("\n\u2705 Master Vector Generated Successfully.")

c:\Users\rajti\raj123\envs\clearsight\lib\site-packages\insightface\utils\transform.py:68: FutureWarning: `rcond` parameter will change to the default of machine precision times ``max(M, N)`` where M and N are the input matrix dimensions.
To use the future default and silence this warning we advise to pass `rcond=None`, to keep using the old, explicitly pass `rcond=-1`.
  P = np.linalg.lstsq(X_homo, Y)[0].T # Affine matrix. 3 x 4


Extracted vector from C:/Users/rajti/Downloads/srk1.webp
Extracted vector from C:/Users/rajti/Downloads/srk2.jpg
Extracted vector from C:/Users/rajti/Downloads/srk3.jpg

✅ Master Vector Generated Successfully.


### Phase 3: Tracklet Generation (The Deep Scan)
Instead of verifying identities blindly on every frame, we scan the entire video using ByteTrack. We build a database of every single person (a "Tracklet") and only extract the absolute clearest face they show during their entire time on screen.

In [3]:
video_path = 'C:/Users/rajti/Downloads/footage1.mp4'
cap = cv2.VideoCapture(video_path)
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
fps = cap.get(cv2.CAP_PROP_FPS)

# Tracklet Database: 
# { track_id: { 'boxes': {frame_idx: [x1,y1,x2,y2]}, 'best_face_img': array, 'max_face_size': int } }
tracklets = {}

print(f"Scanning {total_frames} frames to build Tracklet Database...")

results = yolo_model.track(source=video_path, classes=[0], stream=True, persist=True, verbose=False, tracker="bytetrack.yaml")

frame_idx = 0
for r in results:
    frame_bgr = r.orig_img
    
    if r.boxes.id is not None:
        boxes = r.boxes.xyxy.cpu().numpy()
        track_ids = r.boxes.id.cpu().numpy().astype(int)
        
        for box, track_id in zip(boxes, track_ids):
            if track_id not in tracklets:
                tracklets[track_id] = {'boxes': {}, 'best_face_img': None, 'max_face_size': 0}
            
            # Save the exact box location for this frame so we can retroactively draw it later!
            tracklets[track_id]['boxes'][frame_idx] = box
            
            # Extract body crop to search for a face
            x1, y1, x2, y2 = box.astype(int)
            x, y = max(0, x1), max(0, int(y1 - (y2-y1)*0.2)) # Expand up for head
            w, h = x2 - x1, y2 - y1
            
            body_crop = frame_bgr[y:y+h, x:x+w]
            if body_crop.size == 0: continue
            
            # Quick face detection on the crop
            faces = face_app.get(body_crop)
            if len(faces) > 0:
                best_face = max(faces, key=lambda f: (f.bbox[2]-f.bbox[0])*(f.bbox[3]-f.bbox[1]))
                face_area = (best_face.bbox[2]-best_face.bbox[0]) * (best_face.bbox[3]-best_face.bbox[1])
                
                # If this is the clearest/closest we've seen this person's face, save it!
                if face_area > tracklets[track_id]['max_face_size']:
                    tracklets[track_id]['max_face_size'] = face_area
                    tracklets[track_id]['best_face_img'] = body_crop.copy()
                    tracklets[track_id]['best_face_embedding'] = best_face.embedding

    frame_idx += 1
    if frame_idx % 50 == 0:
        print(f"Processed {frame_idx}/{total_frames} frames...")

cap.release()
print(f"\u2705 Tracklet Database built! Found {len(tracklets)} unique people in the footage.")

Scanning 195 frames to build Tracklet Database...


c:\Users\rajti\raj123\envs\clearsight\lib\site-packages\insightface\utils\transform.py:68: FutureWarning: `rcond` parameter will change to the default of machine precision times ``max(M, N)`` where M and N are the input matrix dimensions.
To use the future default and silence this warning we advise to pass `rcond=None`, to keep using the old, explicitly pass `rcond=-1`.
  P = np.linalg.lstsq(X_homo, Y)[0].T # Affine matrix. 3 x 4


Processed 50/195 frames...
Processed 100/195 frames...
Processed 150/195 frames...
✅ Tracklet Database built! Found 42 unique people in the footage.


### Phase 4: Z-Score Statistical Anomaly Thresholding
Instead of guessing a threshold, we analyze the similarity scores of every person in the video. The true target will stand out as a massive statistical outlier.

In [4]:
scores = {}

def cosine_sim(a, b):
    a = a / np.linalg.norm(a)
    b = b / np.linalg.norm(b)
    return np.dot(a, b)

for track_id, data in tracklets.items():
    if data.get('best_face_embedding') is not None:
        score = cosine_sim(master_vector, data['best_face_embedding'])
        scores[track_id] = score

if not scores:
    raise ValueError("No faces detected in the entire video!")

# Extract raw scores to find the anomaly
score_values = list(scores.values())
mean_score = np.mean(score_values)
std_score = np.std(score_values) if np.std(score_values) > 0 else 0.01

print("--- STATISTICAL ANALYSIS ---")
print(f"Crowd Average Similarity: {mean_score:.3f}")

# Find the absolute best match
best_id = max(scores, key=scores.get)
best_score = scores[best_id]
z_score = (best_score - mean_score) / std_score

print(f"Peak Target Similarity (Body #{best_id}): {best_score:.3f}")
print(f"Z-Score Anomaly Rating: {z_score:.2f} standard deviations above normal")

# If Z-Score > 2.0, they are a mathematical outlier (a verified match!)
TARGET_IDS = set()
if z_score >= 1.5 or best_score >= 0.25:
    print(f"\n\ud83c\udfaf POSITIVE ID: Suspect is mathematically verified as Tracklet #{best_id}!")
    TARGET_IDS.add(best_id)
    
    # Check if there are other Tracklets that are ALSO the suspect (if YOLO lost tracking and assigned a new ID)
    DYNAMIC_THRESHOLD = max(mean_score + (2.5 * std_score), best_score - 0.05)
    print(f"Auto-calibrated dynamic threshold to {DYNAMIC_THRESHOLD:.3f}")
    
    for track_id, score in scores.items():
        if track_id != best_id and score >= DYNAMIC_THRESHOLD:
            print(f"\ud83c\udfaf POSITIVE ID: Secondary Tracklet #{track_id} also verified as Suspect!")
            TARGET_IDS.add(track_id)
else:
    print("\n\u274c NEGATIVE ID: The suspect is NOT in this video. No statistical anomaly found.")

Exception in callback BaseAsyncIOLoop._handle_events(1160, 1)
handle: <Handle BaseAsyncIOLoop._handle_events(1160, 1)>
Traceback (most recent call last):
  File "c:\Users\rajti\raj123\envs\clearsight\lib\site-packages\jupyter_client\session.py", line 95, in json_packer
    return json.dumps(
UnicodeEncodeError: 'utf-8' codec can't encode characters in position 199-200: surrogates not allowed

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "c:\Users\rajti\raj123\envs\clearsight\lib\asyncio\events.py", line 80, in _run
    self._context.run(self._callback, *self._args)
  File "c:\Users\rajti\raj123\envs\clearsight\lib\site-packages\tornado\platform\asyncio.py", line 208, in _handle_events
    handler_func(fileobj, events)
  File "c:\Users\rajti\raj123\envs\clearsight\lib\site-packages\zmq\eventloop\zmqstream.py", line 600, in _handle_events
    self._handle_recv()
  File "c:\Users\rajti\raj123\envs\clearsight\lib\site-packag

### Phase 5: Retroactive Rendering (Backtracking)
Because we saved the exact bounding boxes in the Tracklet Database during Phase 3, we don't even need to run Neural Networks again! We simply replay the video and draw the boxes on the target from the exact second they appeared.

In [5]:
if not TARGET_IDS:
    print("No target to render.")
else:
    # Combine all known frame boxes for the target(s)
    target_frames = {}
    for tid in TARGET_IDS:
        for frame_idx, box in tracklets[tid]['boxes'].items():
            target_frames[frame_idx] = box
            
    print(f"Rendering final video. Suspect is visible in {len(target_frames)} frames.")
    
    cap = cv2.VideoCapture(video_path)
    out_path = 'C:/Users/rajti/Downloads/Projects/ACADEMIC INTERNSHIP/ClearSight_Project/ClearSight_V6_Output.mp4'
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(out_path, fourcc, fps, (int(cap.get(3)), int(cap.get(4))))
    
    frame_idx = 0
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret: break
        
        if frame_idx in target_frames:
            x1, y1, x2, y2 = target_frames[frame_idx].astype(int)
            cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 4)
            cv2.putText(frame, "TARGET ACQUIRED", (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 255, 0), 3)
            
        out.write(frame)
        frame_idx += 1
        
        if frame_idx % 50 == 0:
            print(f"Rendered {frame_idx}/{total_frames} frames...")
            
    cap.release()
    out.release()
    print(f"\n\u2705 RENDERING COMPLETE! Saved to {out_path}")

Rendering final video. Suspect is visible in 12 frames.
Rendered 50/195 frames...
Rendered 100/195 frames...
Rendered 150/195 frames...

✅ RENDERING COMPLETE! Saved to C:/Users/rajti/Downloads/Projects/ACADEMIC INTERNSHIP/ClearSight_Project/ClearSight_V6_Output.mp4
